# Exercise 1 - Space-Filling Curves and Space Partitioning [90-120min]

For the first exercise in the course Spatial Databases (WiSe 26/27), we will take a closer look at space-filling curves and the space-partitioning index.

Most of the exercises will follow a similar scheme: one part will be a coding task, and a second part will be a mix of multiple-choice and free-text questions.

You must submit this exercise by <span style='color: #FF746C' id='deadline'>**03.11.2026 16:00h**</span>. Upload this file with your changes to Moodle. We will discuss the solutions in class before introducing the exercise for the following week(s).

To be eligible to take the exam, you must achieve at least 50% of all available points over all exercise sessions.

Only modify the notebook cells you are requested to. If you are unsure how to solve a task, set up the environment, or suspect a mistake, do not hesitate to contact `maximilian.schulze@hpi.de` via email or `Maximilian Schulze` via HPI's Slack. You can also post questions that are of interest to everyone to Moodle's forum and we will answer them there.

In [ ]:
from utils import (
    SIZE,
    IndexComparisonPlot,
    generate_city_data,
    plot_curve_index,
    plot_dataset,
    plot_distance_pairs,
    plot_spatial_curve,
)

%matplotlib inline

## Space-Filling Curves

Space-filling curves map spatial data to a 1D index with the goal of preserving spatial neighborhood.

The most common curves are the Z-curve and the Hilbert-curve. The code of a function that maps a 2D point to its Hilbert-curve index is given below.

In [ ]:
def hilbert_curve_index(x: int, y: int, size: int = 16) -> int:
    """Convert a 2D point to a Hilbert curve index.
    
    Args:
        x (int): The x-coordinate of the point.
        y (int): The y-coordinate of the point.
        size (int): The size of the Hilbert curve (must be a power of
            2). Default is 16.

    Returns:
        int: The Hilbert curve index of the point.
    """
    n_bits = size.bit_length() - 1
    pair_to_digit = {(0, 0): 0, (0, 1): 1, (1, 0): 3, (1, 1): 2}
    digits = [pair_to_digit[(x >> i) & 1, (y >> i) & 1] for i in reversed(range(n_bits))]

    for i, digit in enumerate(digits):
        if digit == 0:
            swap = {1: 3, 3: 1}
        elif digit == 3:
            swap = {0: 2, 2: 0}
        else:
            continue
        for j in range(i + 1, n_bits):
            digits[j] = swap.get(digits[j], digits[j])

    index = 0
    for digit in digits:
        index = 4 * index + digit
    return index

In [ ]:
plot_spatial_curve(hilbert_curve_index, 16, 16, "Hilbert-curve", ylim=(0, 8))

<span style='color: #FF746C' id='task1.1'>**Task 1.1 [15pts]:**</span> Implement a function that maps `x` and `y` coordinates to its corresponding Z-curve index. The leading, most significant bit should be from `y`. You can view a plot of your curve function below your implementation. For the input parameters assume $x,y \in \mathbb{N}_0$.

In [ ]:
def z_curve_index(x: int, y: int) -> int:
    """Converts a 2D point to a Z-curve index.
    
    Args:
        x (int): The x-coordinate of the point.
        y (int): The y-coordinate of the point.
        
    Returns:
        int: The Z-curve index corresponding to the (x, y) point.
    """
    return 0

In [ ]:
plot_spatial_curve(z_curve_index, 16, 16, "Z-curve", ylim=(0, 8))

## Hilbert-Curve and Z-Curve on Clustered Data

In [ ]:
curves = {
    "Z-curve": z_curve_index,
    "Hilbert curve": lambda x, y: hilbert_curve_index(x, y, SIZE),
}

Now, we will look at how your implementation of the Z-curve index and the Hilbert-curve index map 2D points into a 1D space.

For that, we create a synthetic dataset with points in Germany and a higher density around some cities.
We also sample some query points from the same distribution.

In [ ]:
city_dataset = set(generate_city_data(4096, seed=1))
city_query_points = set(generate_city_data(128, seed=2)) - city_dataset
plot_dataset(city_dataset, city_query_points)

The following plot colors each sampled point by its curve index. Sharp color contrasts indicate large jumps in the 1D index.

In [ ]:
plot_curve_index(city_dataset, curves)

The following plot compares, for each query point and its 16 nearest dataset points, the Euclidean distance with the curve distance (the absolute difference of the curve indices). Pairs that lie in different cells of a 4×4 grid over the domain (quadrants) are shown in orange.

In [ ]:
plot_distance_pairs(city_dataset, city_query_points, curves)

The orange pairs with a Euclidean distance of a few hundred units and a curve distance of more than $10^9$ are, for example, from around Dresden, where the index jumps for both the Hilbert curve and the Z-curve.

### Z-Curve-Based Operators

Now, we look at how to use the Z-curve for point and rectangle queries. The following class definition uses your Z-curve implementation.

<span style='color: #FF746C' id='task1.2'>**Task 1.2 [5pts]:**</span> Implement the function `ZCurveIndex.has_point`. The runtime of the search should be sub-linear in the number of stored points. Use `self.keys` and `binary_search`. Do not use `self.points`.

In [ ]:
def binary_search(sorted_list: list[int], target: int, type="left") -> int:
    """Perform a binary search on a sorted list and return the index of the target, or
    the index where it should be inserted if not found.
    
    Args:
        sorted_list (list[int]): A list of integers sorted in ascending order.
        target (int): The integer to search for.
        type (str): "left" to return the left insertion point, "right" for the right
            insertion point.
    
    Returns:
        int: The index of the target if found (for "right" the index after it), or the insertion point if not found.
    """
    left, right = 0, len(sorted_list) - 1
    while left <= right:
        mid = (left + right) // 2
        if sorted_list[mid] < target:
            left = mid + 1
        elif sorted_list[mid] > target:
            right = mid - 1
        else:
            return mid if type == "left" else mid + 1
    return left

In [ ]:
class ZCurveIndex:
    def __init__(self, points: set[tuple[int, int]]):
        """Initialize the ZCurveIndex with a set of points.
        
        Args:
            points (set[tuple[int, int]]): A set of points to index.
        """
        curve = z_curve_index
        self.curve = curve
        self.points = points
        keyed_points = sorted((curve(*point), point) for point in points)
        self.keys = [key for key, _ in keyed_points]
        self.indexed_points = [point for _, point in keyed_points]

    def has_point(self, point: tuple[int, int]) -> bool:
        """Indicates whether the given point is in the index.
        
        Args:
            point (tuple[int, int]): The point to check for existence in the index.

        Returns:
            bool: True if the point is in the index, False otherwise.
        """

    def points_in_rectangle(self, xlow: int, xhigh: int, ylow: int, yhigh: int) -> list[tuple[int, int]]:
        """Returns all points in the index that are within the given rectangle defined by the lower
        left corner (xlow, ylow) and the upper right corner (xhigh, yhigh).
        
        Args:
            xlow (int): The x-coordinate of the lower left corner of the rectangle.
            xhigh (int): The x-coordinate of the upper right corner of the rectangle.
            ylow (int): The y-coordinate of the lower left corner of the rectangle.
            yhigh (int): The y-coordinate of the upper right corner of the rectangle.
            
        Returns:
            list[tuple[int, int]]: A list of points in the index that are within the given rectangle.
        """
        start = binary_search(self.keys, self.curve(xlow, ylow), type="left")
        end = binary_search(self.keys, self.curve(xhigh, yhigh), type="right")
        return [p for p in self.indexed_points[start:end] if xlow <= p[0] <= xhigh and ylow <= p[1] <= yhigh]


## The Uniform Grid Index

<span style='color: #FF746C' id='task1.3'>**Task 1.3 [15pts]:**</span> Finish the implementation of the `UniformGridIndex`, a uniform grid index for 2D point data. Assume the data to be static, i.e., you do not need to handle insertions and deletions. For this exercise we only need two operations: `has_point`, which checks whether a given point is stored in the index, and `points_in_rectangle`, the range query, which takes the minimum and maximum coordinates of a query rectangle (borders inclusive) and returns all points within it.

The spatial extent of the index is the bounding box of the dataset, i.e., $[x_{min}, x_{max}] \times [y_{min}, y_{max}]$. It is divided into `divisions_per_axis` $\times$ `divisions_per_axis` equally wide cells. Along the x-axis, a point lies in cell $\lfloor (x - x_{min}) \cdot d / (x_{max} - x_{min} + 1) \rfloor$ with $d$ = `divisions_per_axis`. The y-axis works the same way. Query rectangles and points passed to `has_point` may lie partly or completely outside this extent. Your implementation must handle this without errors.

In [ ]:
class UniformGridIndex:
    def __init__(self, points: set[tuple[int, int]], divisions_per_axis: int):
        """Initialize the uniform grid index with a set of points and a number of divisions per axis."""

    def has_point(self, point: tuple[int, int]) -> bool:
        """Check if the point is stored in the index.
        Find the cell that contains the point and check if the point is in that cell.
        
        Args:
            point (tuple[int, int]): The point to check for existence in the index.
            
        Returns:
            bool: True if the point is in the index, False otherwise.
        """

    def points_in_rectangle(self, xlow: int, xhigh: int, ylow: int, yhigh: int) -> list[tuple[int, int]]:
        """Returns all points in the index that are within the given rectangle defined by the lower
        left corner (xlow, ylow) and the upper right corner (xhigh, yhigh).
        
        Args:
            xlow (int): The x-coordinate of the lower left corner of the rectangle.
            xhigh (int): The x-coordinate of the upper right corner of the rectangle.
            ylow (int): The y-coordinate of the lower left corner of the rectangle.
            yhigh (int): The y-coordinate of the upper right corner of the rectangle.
            
        Returns:
            list[tuple[int, int]]: A list of points in the index that are within the given rectangle.
        """


### Comparing the Uniform Grid and the Z-Curve Index

Now you can see the difference between the Z-Curve based index and the uniform grid index in action.

Once your index is implemented, use the slider to see how the number of divisions per axis partitions the German cities dataset. The grid spans the extent of the dataset, as required above.

The upper row shows how both indexes organize the data. On the left, every cell is colored by the number of points your `points_in_rectangle` returns for it. Cells whose answer differs from the points that actually lie in them are outlined in red, and the title shows how many dataset points your `has_point` finds. On the right, every point is colored by its Z-curve index.

The lower row runs the same range query on both indexes. Drag on either map to draw a new query rectangle, drag inside the rectangle to move it, or drag its handles to resize it. Blue points are inspected by the index but lie outside of the rectangle. The grid index inspects the points in the cells the rectangle overlaps (light blue), the Z-curve index all points whose key lies between the keys of the lower left and the upper right corner of the rectangle.

In [ ]:
IndexComparisonPlot(city_dataset, UniformGridIndex, ZCurveIndex)

## Written Questions [43pts]

**<span style='color: #FF746C' id='task1.4'>Task 1.4 [24pts]:</span> Solve the following tasks. Write your answers in the answer cell. Do not modify the question cells or add any new cells.**

**Example:**

What are best pizza toppings?

1. Mozzarella
2. Pineapple
3. Tomato-Sauce

**Answer:** 1,3


<span style='color: #FF746C' id='task1.4.a'>**(a) [4pts]:**</span> Assume raster data. For each of the following operations, is it a local, focal, zonal or global operation? Explain each answer in one to two sentences.

1. Create a boolean mask that marks every land cell adjacent to a water cell (coast cells).
2. Find all cells with an elevation above 3 km.
3. Compute the average elevation of each Bundesland.
4. Compute, for every cell, the distance to the nearest lake.

**Answer:**

| Query | Operation Type | Explanation |
| ----- | -------------- | ----------- |
|   1.  | \<your answer here\> | \<your explanation here\> |
|   2.  | \<your answer here\> | \<your explanation here\> |
|   3.  | \<your answer here\> | \<your explanation here\> |
|   4.  | \<your answer here\> | \<your explanation here\> |

<span style='color: #FF746C' id='task1.4.b'>**(b) [2pts]:**</span> Which of the following statements is/are true?
1. Computing the Z-curve index of a coordinate pair is computationally cheaper than computing the Hilbert-curve index.
2. Computing the Hilbert-curve index of a coordinate pair is computationally cheaper than computing the Z-curve index.

**Answer:** \<your answer here\>

<span style='color: #FF746C' id='task1.4.c'>**(c) [2pts]:**</span> Which of the following statements is/are true in general (for grids larger than $2\times2$)?
1. Consecutive Z-curve indices are always directly adjacent cells (horizontal, vertical or diagonal).
2. Consecutive Hilbert-curve indices are always directly adjacent cells (horizontal, vertical or diagonal).
3. Directly adjacent cells (horizontal, vertical or diagonal) always have consecutive Z-curve indices.
4. Directly adjacent cells (horizontal, vertical or diagonal) always have consecutive Hilbert-curve indices.

**Answer:** \<your answer here\>

<span style='color: #FF746C' id='task1.4.d'>**(d) [4pts]:**</span> Answer the following question briefly.

As seen above in the curve index plots of sampled points in Germany, points in Dresden have strong color discrepancies, i.e., large curve distances to each other. Why is it problematic if spatially close items are numerically far apart after converting from 2D to 1D?

**Answer:** \<your answer here\>

<span style='color: #FF746C' id='task1.4.e'>**(e) [6pts]:**</span> Answer the following question briefly and reason about your solution.

In a constrained integer space of size $128\times128$, a quadtree is built with a maximum leaf capacity of 16. A node is split if it holds more than 16 points. How many leaf cells (including empty ones) does the quadtree store in the **worst case** if 48 unique points are inserted? Unique means no two points share their position. Worst case refers to the **maximum** number of leaf cells. Assume the minimum cell size is $1\times1$.

**Answer:** \<your answer here\>

<span style='color: #FF746C' id='task1.4.f'>**(f) [6pts]:**</span> Answer the following question briefly and reason about your solution.

In a constrained integer space of size $128\times128$, a quadtree is built with a maximum leaf capacity of 16. A node is split if it holds more than 16 points. How many leaf cells (including empty ones) does the quadtree store in the **best case** if 48 unique points are inserted? Unique means no two points share their position. Best case refers to the **minimum** number of leaf cells. Assume the minimum cell size is $1\times1$.

**Answer:** \<your answer here\>

<span style='color: #FF746C' id='task1.5'>**Task 1.5 [19pts]:**</span> You are given the following spatial objects:

![Image with objects](exercise1-objects.svg)

<span style='color: #FF746C' id='task1.5.a'>**(a) [4pts]:**</span> Fill out the 9IM for A and B. Rows are the parts of A, columns the parts of B.

**Answer:**

A $\downarrow$, B $\rightarrow$ | Interior | Boundary | Exterior
--- | --- | --- | ---
Interior | ? | ? | ?
Boundary | ? | ? | ?
Exterior | ? | ? | ?

<span style='color: #FF746C' id='task1.5.b'>**(b) [4pts]:**</span> Fill out the DE-9IM for A and B. Rows are the parts of A, columns the parts of B.

**Answer:**

A $\downarrow$, B $\rightarrow$ | Interior | Boundary | Exterior
--- | --- | --- | ---
Interior | ? | ? | ?
Boundary | ? | ? | ?
Exterior | ? | ? | ?

<span style='color: #FF746C' id='task1.5.c'>**(c) [4pts]:**</span> Fill out the 9IM for B and E. Rows are the parts of B, columns the parts of E.

**Answer:**

B $\downarrow$, E $\rightarrow$ | Interior | Boundary | Exterior
--- | --- | --- | ---
Interior | ? | ? | ?
Boundary | ? | ? | ?
Exterior | ? | ? | ?

<span style='color: #FF746C' id='task1.5.d'>**(d) [4pts]:**</span> Fill out the DE-9IM for B and E. Rows are the parts of B, columns the parts of E.

**Answer:**

B $\downarrow$, E $\rightarrow$ | Interior | Boundary | Exterior
--- | --- | --- | ---
Interior | ? | ? | ?
Boundary | ? | ? | ?
Exterior | ? | ? | ?

<span style='color: #FF746C' id='task1.5.e'>**(e) [3pts]:**</span> Name the topological relationships (equals, disjoint, touches, contains, within, overlaps or crosses) for the following pairs:

1. A and B
2. A and C
3. A and D
4. A and E
5. C and B
6. D and A

**Answer:**
1. \<Your answer here\>
2. \<Your answer here\>
3. \<Your answer here\>
4. \<Your answer here\>
5. \<Your answer here\>
6. \<Your answer here\>